## 1. 调用大模型

使用 Python 发送消息并取得大模型的文本答案。

### 1.1. 配置客户端

| 配置 | 作用 |
| --- | --- |
| `LLM_API_KEY` | 身份认证，不能公开或提交到 Git |
| `LLM_BASE_URL` | API 服务地址 |
| `LLM_MODEL` | 模型名称 |

`OpenAI(api_key=..., base_url=...)` 创建客户端并保存认证信息和 API 地址。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None
print(
    f"API 已配置, 配置的模型是{LLM_MODEL}"
    if client
    else "未配置 API：保留本地步骤，调用模型的单元会跳过"
)

API 已配置, 配置的模型是deepseek-v4-flash-0731


### 1.2. 发送请求

`client.chat.completions.create(...)` 创建一次对话回答。

| 参数 | 作用 |
| --- | --- |
| `model` | 指定模型 |
| `messages` | 按顺序提交包含 `role` 和 `content` 的消息 |
| `temperature` | 控制输出随机性，事实问答通常设为 `0` |

API 返回包含请求编号、用量和候选答案的响应对象。答案正文位于：

```python
response.choices[0].message.content
```

In [2]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "用两句话解释什么是向量检索。"}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print("请配置 LLM_API_KEY 或 OPENAI_API_KEY")

向量检索是将文本、图片等数据转换为向量表示，通过计算向量之间的距离（如余弦相似度）来查找最相似内容的技术。它能在大规模候选集中快速返回语义相近的结果，常用于搜索、推荐和问答系统。


### 1.3. 返回两条候选答案

模型服务支持 `n` 参数时，设置 `n=2` 可针对同一个问题返回两条候选答案。

- **创意生成**：生成多个标题、广告语或商品描述。
- **程序筛选**：由规则、评分模型或人工选择答案。
- **投票校验**：根据多条推理结果的一致性选择结论。

多生成候选会增加输出量和费用；普通聊天和 RAG 事实问答通常使用 `n=1`。百炼在开启思考模式时要求 `n=1`，因此本课配置通过 `extra_body={"enable_thinking": False}` 关闭思考模式。其他服务应删除这个百炼专用参数。

In [3]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "用一句话解释什么是 RAG。"}],
        temperature=1,
        n=2,
        extra_body={"enable_thinking": False},
    )
    for index, choice in enumerate(response.choices, start=1):
        print(f"候选答案 {index}：{choice.message.content}")
else:
    print("请配置 LLM_API_KEY 或 OPENAI_API_KEY")


候选答案 1：RAG（检索增强生成）是一种让大模型在生成回答前先从一个外部知识库中检索相关信息，再把检索到的内容作为参考依据来生成答案的技术。
候选答案 2：RAG（检索增强生成）是一种让大模型先在外部知识库中查找相关信息，再基于这些信息来生成回答的技术，从而有效解决模型“胡编乱造”和知识过时的问题。


### 1.4. 设置消息角色

`system` 规定模型的回答方式，`user` 提出具体问题。RAG 的“只能根据资料回答”通常放在 system 消息里。


In [4]:
messages = [
    {"role": "system", "content": "你是服饰箱包知识库助手，只根据用户提供的资料回答。"},
    {"role": "user", "content": "SKU-JK902 是什么产品？"},
]

if client:
    response = client.chat.completions.create(
        model=LLM_MODEL, messages=messages, temperature=0
    )
    print(response.choices[0].message.content)
else:
    print(messages)

我目前没有关于“SKU-JK902”的资料，无法回答。请提供相关产品资料。


### 1.5. 认识生成参数

| 参数 | 作用 | 常见用法 |
| --- | --- | --- |
| `temperature` | 控制输出随机性 | 事实问答用较低值，创意写作用较高值 |
| `top_p` | 从累计概率达到指定比例的候选中选择 | 控制回答多样性 |
| `top_k` | 每一步保留概率最高的 K 个候选 | K 越小通常越保守；部分接口不支持 |
| `max_tokens` | 限制生成的 Token 数量 | 控制回答长度、时间和费用 |

通常保持 `top_p` 和 `top_k` 默认值，先调整 `temperature`。本课程使用 `temperature=0`，以降低事实问答的输出随机性。